# Biological BMI Paper — LASSO Modeling for Time-series BMI with Sex-stratified Version

***by Kengo Watanabe***  

In this notebook, LASSO models for BMI (biological BMIs) are generated from the time-series Arivale datasets. To avoid the participant bias (e.g., observation times are different between the participants), only the baseline datasets are used for training the models, while generating both sex model and sex-stratified models.  
–> Subsequently, longitudinal change of BMI and biological BMIs are analyzed with linear mixed model (LMM). Because of heavy loading, the subsequent part is divided into another notebook.  

Input:  
* Cleaned time-series metabolomics: 210104_Biological-BMI-paper_RF-imputation_time-series-metDF-with-RF-imputation.tsv  
* Baseline MetBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv  
* Baseline MetBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv  
* Baseline MetBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv  
* Cleaned time-series proteomics: 210104_Biological-BMI-paper_RF-imputation_time-series-protDF-with-RF-imputation.tsv  
* Baseline ProtBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv  
* Baseline ProtBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv  
* Baseline ProtBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv  
* Cleaned time-series clinical labs: 210104_Biological-BMI-paper_RF-imputation_time-series-chemDF-with-RF-imputation.tsv  
* Baseline ChemBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv  
* Baseline ChemBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv  
* Baseline ChemBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv  
* Cleaned time-series combined omics: 210104_Biological-BMI-paper_RF-imputation_time-series-combiDF-with-RF-imputation.tsv  
* Baseline CombiBMI predictions: 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv  
* Baseline CombiBMI predictions (female model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv  
* Baseline CombiBMI predictions (male model): 210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv  
* Summary of all biological BMIs (baseline): 210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv  

Output:  
* Intermediate tables for other notebook (BMI predictions)  

Original notebook:  
* 210106_Biological-BMI-paper_longitudinal-LASSO.ipynb  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

!conda list

## 1. Metabolomics

### 1-1. Prepare the baseline and sex-dependent DFs for LASSO

In [ ]:
#Import the cleaned time-series DF
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_time-series-metDF-with-RF-imputation.tsv'
tsDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF)

In [ ]:
#Prepare the baseline DF for LASSO
tempDF = tsDF.sort_values(by=['public_client_id', 'days_in_program'], ascending=True)
baseDF = tempDF.drop_duplicates('public_client_id', keep='first')
display(baseDF)

In [ ]:
#Split DF for sex-dependent models
baseDF_F = baseDF.loc[baseDF['Sex']=='F']
baseDF_M = baseDF.loc[baseDF['Sex']=='M']
baseDF_B = baseDF#Not copy just rename
print('Baseline DF nrows (unique ID): (Female, Male, Both) = (',
      len(baseDF_F), ',', len(baseDF_M), ',', len(baseDF_B), ')')
tsDF_F = tsDF.loc[tsDF['Sex']=='F']
tsDF_M = tsDF.loc[tsDF['Sex']=='M']
tsDF_B = tsDF#Not copy just rename
print('Time-series DF nrows: (Female, Male, Both) = (',
      len(tsDF_F), ',', len(tsDF_M), ',', len(tsDF_B), ')')
print('Time-series DF unique ID: (Female, Male, Both) = (',
      len(tsDF_F['public_client_id'].unique()), ',',
      len(tsDF_M['public_client_id'].unique()), ',',
      len(tsDF_B['public_client_id'].unique()), ')')

### 1-2. Standarization with the baseline DF

In [ ]:
#Female cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_F.loc[:, ~baseDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_F.loc[:, ~tsDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Male cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_M.loc[:, ~baseDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_M.loc[:, ~tsDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_B.loc[:, ~baseDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_B.loc[:, ~tsDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 1-3. A cross-validated LASSO model trained with the baseline measurement

> For the baseline analyses, 10 LASSO models with 10 cross-validation were generated from 10 splitted DF.  
> ***–> Because it is difficult to manage the prediction difference b/w models and b/w participants, generate only 1 LASSO model with 10 CV.***  
> <– There are participants with only the baseline measurement, which would relieve the overfitting for the cohort in time-series analyses.  

In [ ]:
#Female model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_F.drop(columns=list(set(baseDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_F_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_F = model.score(baseDF_F_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_F_zx), index=tempDF.index, name='log_BaseMetBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseMetBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_F = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_F[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_F)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_F_zx), index=tsDF_F.index, name='log_MetBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='MetBMI')
tempDF = tsDF_F[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_F = pd.merge(tempDF, baseDF_F, on='public_client_id', how='left')
tsDF_F['KeyIndex'] = tsDF_F['public_client_id'] + '_day' + tsDF_F['days_in_program'].astype(str)
tsDF_F = tsDF_F.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_F)

In [ ]:
#Male model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_M.drop(columns=list(set(baseDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_M_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_M = model.score(baseDF_M_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_M_zx), index=tempDF.index, name='log_BaseMetBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseMetBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_M = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_M[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_M)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_M_zx), index=tsDF_M.index, name='log_MetBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='MetBMI')
tempDF = tsDF_M[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_M = pd.merge(tempDF, baseDF_M, on='public_client_id', how='left')
tsDF_M['KeyIndex'] = tsDF_M['public_client_id'] + '_day' + tsDF_M['days_in_program'].astype(str)
tsDF_M = tsDF_M.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_M)

In [ ]:
#Both sex model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_B.drop(columns=list(set(baseDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_B_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_B = model.score(baseDF_B_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_B_zx), index=tempDF.index, name='log_BaseMetBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseMetBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_B = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_B[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_B)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_B_zx), index=tsDF_B.index, name='log_MetBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='MetBMI')
tempDF = tsDF_B[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_B = pd.merge(tempDF, baseDF_B, on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_B)

### 1-4. Prediction accuracy of LASSO model

In [ ]:
#Plot R2 score
tempDF = pd.DataFrame([R2_F, R2_M, R2_B], index=['Female', 'Male', 'Both sex'], columns=['BaseMetBMI'])
tempDF.index.set_names('Model', inplace=True)
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF.reset_index(), y='Model', x='BaseMetBMI',
            hue='Model', palette='Set1', dodge=False, edgecolor='black')
sns.despine()
plt.xlabel('In-sample '+r'$R^2$'+' in 1 LASSO model')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

print('Female model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_F)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_F['log_BaseBMI'], baseDF_F['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_F['BaseBMI'], baseDF_F['BaseMetBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_F[['log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseMetBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseMetBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_F[['log_MetBMI', 'MetBMI', 'log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Male model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_M)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_M['log_BaseBMI'], baseDF_M['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_M['BaseBMI'], baseDF_M['BaseMetBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_M[['log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseMetBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseMetBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_M[['log_MetBMI', 'MetBMI', 'log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Both sex model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_B)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_B['log_BaseBMI'], baseDF_B['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_B['BaseBMI'], baseDF_B['BaseMetBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_B[['log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseMetBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseMetBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_B[['log_MetBMI', 'MetBMI', 'log_BaseMetBMI', 'BaseMetBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

In [ ]:
print('Baseline biological BMI:')
#Plot all models for the baseline
tempDF = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseMetBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Biological BMI [kg/m'+r'$^2$'+']')
plt.ylabel('Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
#Plot difference b/w sex-combined and sex-stratified models
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF = tempDF.set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseMetBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()
print('')

#Plot difference b/w previous 10 models and current 1 model
print('Comparison b/w the previous 10 models and the current 1 model for the baseline:')
##Prepare the previous predictions
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv'
tempDF3 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])
tempDF = tempDF.reset_index()
tempDF1 = tempDF[['public_client_id', 'BaseBMI', 'Model', 'BaseMetBMI']]
##Prepare the current predictions
tempDF2 = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF2['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF2 = tempDF2.reset_index()
tempDF2 = tempDF2[['public_client_id', 'BaseBMI', 'Model', 'BaseMetBMI']]
##Merge
tempDF = pd.concat([tempDF1, tempDF2], axis=0)
tempDF['Modeling'] = np.repeat(['Previous', 'Current'], [len(tempDF1), len(tempDF2)])
##Plot1
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='BaseMetBMI', y='BaseBMI', hue='Modeling', palette='tab10',
               col='Model', col_wrap=3, scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
#p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax in p.axes.ravel():
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
p.set_titles('{col_name}')
p.set_axis_labels('Biological BMI [kg/m'+r'$^2$'+']', 'Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
##Plot2
tempDF = tempDF.set_index(['public_client_id', 'Model']).pivot(columns='Modeling')['BaseMetBMI']
tempDF = tempDF.reset_index()
tempL = ['Female', 'Male', 'Both sex']
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Previous', y='Current',
               hue='Model', hue_order=tempL, palette='Set1', legend=False,
               col='Model', col_order=tempL, col_wrap=3,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, model in zip(p.axes.ravel(), tempL):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Model']==model]['Previous']
    tempS2 = tempDF.loc[tempDF['Model']==model]['Current']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
p.set_titles('{col_name}')
p.set_axis_labels('10 model predictions [kg/m'+r'$^2$'+']',
                  '1 model predictions [kg/m'+r'$^2$'+']')
plt.show()
print('')

print('Time-series biological BMI:')
#Plot difference b/w sex-combined and sex-stratified models for the time-series
tempDF = pd.concat([tsDF_F, tsDF_M, tsDF_B], axis=0)
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(tsDF_F), len(tsDF_M), len(tsDF_B)])
tempDF = tempDF.reset_index().set_index(['KeyIndex', 'Sex']).pivot(columns='Model')['MetBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()

### 1-5. Baseline obesity classification

In [ ]:
#Sex-combined model
print('Sex-combined model')
tempDF = baseDF_B#Not copy just rename
for col_n in ['BaseBMI', 'BaseMetBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseMetBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempS = tempDF1['BaseMetBMI_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_B = pd.merge(tsDF_B, tempDF[['public_client_id', 'BaseBMI_class', 'BaseMetBMI_class']],
                  on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
tsDF_B = tsDF_B[['public_client_id', 'days_in_program', 'Season', 'log_MetBMI', 'MetBMI',
                 'log_BaseMetBMI', 'BaseMetBMI', 'BaseMetBMI_class',
                 'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_B)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-metBMI-BothSex.csv'
tsDF_B.to_csv(fileDir+fileName, index=True)

In [ ]:
#Sex-stratified model
print('Sex-stratified model')
tempDF = pd.concat([baseDF_F, baseDF_M], axis=0)
for col_n in ['BaseBMI', 'BaseMetBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseMetBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = pd.concat([tempDF1, tempDF2], axis=0)
col_n = 'BaseMetBMI'
tempL = []
for value in tempDF1[col_n].tolist():
    if np.isnan(value):
        tempL.append('NotCalculated')
    elif value < 18.5:
        tempL.append('Underweight')
    elif value < 25:
        tempL.append('Normal')
    elif value < 30:
        tempL.append('Overweight')
    elif value >= 30:
        tempL.append('Obese')
    else:#Just in case
        tempL.append('Error?')
tempDF1[col_n+'_class'] = tempL
tempS = tempDF1[col_n+'_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_FM = pd.concat([tsDF_F, tsDF_M], axis=0)
tsDF_FM = pd.merge(tsDF_FM, tempDF[['public_client_id', 'BaseBMI_class', 'BaseMetBMI_class']],
                   on='public_client_id', how='left')
tsDF_FM['KeyIndex'] = tsDF_FM['public_client_id'] + '_day' + tsDF_FM['days_in_program'].astype(str)
tsDF_FM = tsDF_FM.set_index('KeyIndex')
tsDF_FM = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'log_MetBMI', 'MetBMI',
                   'log_BaseMetBMI', 'BaseMetBMI', 'BaseMetBMI_class',
                   'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                   'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_FM)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-metBMI-FemaleMale.csv'
tsDF_FM.to_csv(fileDir+fileName, index=True)

## 2. Proteomics

### 2-1. Prepare the baseline and sex-dependent DFs for LASSO

In [ ]:
#Import the cleaned time-series DF
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_time-series-protDF-with-RF-imputation.tsv'
tsDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF)

In [ ]:
#Prepare the baseline DF for LASSO
tempDF = tsDF.sort_values(by=['public_client_id', 'days_in_program'], ascending=True)
baseDF = tempDF.drop_duplicates('public_client_id', keep='first')
display(baseDF)

In [ ]:
#Split DF for sex-dependent models
baseDF_F = baseDF.loc[baseDF['Sex']=='F']
baseDF_M = baseDF.loc[baseDF['Sex']=='M']
baseDF_B = baseDF#Not copy just rename
print('Baseline DF nrows (unique ID): (Female, Male, Both) = (',
      len(baseDF_F), ',', len(baseDF_M), ',', len(baseDF_B), ')')
tsDF_F = tsDF.loc[tsDF['Sex']=='F']
tsDF_M = tsDF.loc[tsDF['Sex']=='M']
tsDF_B = tsDF#Not copy just rename
print('Time-series DF nrows: (Female, Male, Both) = (',
      len(tsDF_F), ',', len(tsDF_M), ',', len(tsDF_B), ')')
print('Time-series DF unique ID: (Female, Male, Both) = (',
      len(tsDF_F['public_client_id'].unique()), ',',
      len(tsDF_M['public_client_id'].unique()), ',',
      len(tsDF_B['public_client_id'].unique()), ')')

### 2-2. Standarization with the baseline DF

In [ ]:
#Female cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_F.loc[:, ~baseDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_F.loc[:, ~tsDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Male cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_M.loc[:, ~baseDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_M.loc[:, ~tsDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_B.loc[:, ~baseDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_B.loc[:, ~tsDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 2-3. A cross-validated LASSO model trained with the baseline measurement

> For the baseline analyses, 10 LASSO models with 10 cross-validation were generated from 10 splitted DF.  
> ***–> Because it is difficult to manage the prediction difference b/w models and b/w participants, generate only 1 LASSO model with 10 CV.***  
> <– There are participants with only the baseline measurement, which would relieve the overfitting for the cohort in time-series analyses.  

In [ ]:
#Female model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_F.drop(columns=list(set(baseDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_F_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_F = model.score(baseDF_F_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_F_zx), index=tempDF.index, name='log_BaseProtBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseProtBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_F = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_F[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_F)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_F_zx), index=tsDF_F.index, name='log_ProtBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ProtBMI')
tempDF = tsDF_F[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_F = pd.merge(tempDF, baseDF_F, on='public_client_id', how='left')
tsDF_F['KeyIndex'] = tsDF_F['public_client_id'] + '_day' + tsDF_F['days_in_program'].astype(str)
tsDF_F = tsDF_F.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_F)

In [ ]:
#Male model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_M.drop(columns=list(set(baseDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_M_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_M = model.score(baseDF_M_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_M_zx), index=tempDF.index, name='log_BaseProtBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseProtBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_M = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_M[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_M)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_M_zx), index=tsDF_M.index, name='log_ProtBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ProtBMI')
tempDF = tsDF_M[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_M = pd.merge(tempDF, baseDF_M, on='public_client_id', how='left')
tsDF_M['KeyIndex'] = tsDF_M['public_client_id'] + '_day' + tsDF_M['days_in_program'].astype(str)
tsDF_M = tsDF_M.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_M)

In [ ]:
#Both sex model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_B.drop(columns=list(set(baseDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_B_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_B = model.score(baseDF_B_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_B_zx), index=tempDF.index, name='log_BaseProtBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseProtBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_B = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_B[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_B)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_B_zx), index=tsDF_B.index, name='log_ProtBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ProtBMI')
tempDF = tsDF_B[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_B = pd.merge(tempDF, baseDF_B, on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_B)

### 2-4. Prediction accuracy of LASSO model

In [ ]:
#Plot R2 score
tempDF = pd.DataFrame([R2_F, R2_M, R2_B], index=['Female', 'Male', 'Both sex'], columns=['BaseProtBMI'])
tempDF.index.set_names('Model', inplace=True)
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF.reset_index(), y='Model', x='BaseProtBMI',
            hue='Model', palette='Set1', dodge=False, edgecolor='black')
sns.despine()
plt.xlabel('In-sample '+r'$R^2$'+' in 1 LASSO model')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

print('Female model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_F)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_F['log_BaseBMI'], baseDF_F['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_F['BaseBMI'], baseDF_F['BaseProtBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_F[['log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseProtBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseProtBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_F[['log_ProtBMI', 'ProtBMI', 'log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Male model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_M)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_M['log_BaseBMI'], baseDF_M['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_M['BaseBMI'], baseDF_M['BaseProtBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_M[['log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseProtBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseProtBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_M[['log_ProtBMI', 'ProtBMI', 'log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Both sex model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_B)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_B['log_BaseBMI'], baseDF_B['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_B['BaseBMI'], baseDF_B['BaseProtBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_B[['log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseProtBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseProtBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_B[['log_ProtBMI', 'ProtBMI', 'log_BaseProtBMI', 'BaseProtBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

In [ ]:
print('Baseline biological BMI:')
#Plot all models for the baseline
tempDF = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseProtBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Biological BMI [kg/m'+r'$^2$'+']')
plt.ylabel('Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
#Plot difference b/w sex-combined and sex-stratified models
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF = tempDF.set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseProtBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()
print('')

#Plot difference b/w previous 10 models and current 1 model
print('Comparison b/w the previous 10 models and the current 1 model for the baseline:')
##Prepare the previous predictions
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv'
tempDF3 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])
tempDF = tempDF.reset_index()
tempDF1 = tempDF[['public_client_id', 'BaseBMI', 'Model', 'BaseProtBMI']]
##Prepare the current predictions
tempDF2 = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF2['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF2 = tempDF2.reset_index()
tempDF2 = tempDF2[['public_client_id', 'BaseBMI', 'Model', 'BaseProtBMI']]
##Merge
tempDF = pd.concat([tempDF1, tempDF2], axis=0)
tempDF['Modeling'] = np.repeat(['Previous', 'Current'], [len(tempDF1), len(tempDF2)])
##Plot1
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='BaseProtBMI', y='BaseBMI', hue='Modeling', palette='tab10',
               col='Model', col_wrap=3, scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
#p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax in p.axes.ravel():
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
p.set_titles('{col_name}')
p.set_axis_labels('Biological BMI [kg/m'+r'$^2$'+']', 'Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
##Plot2
tempDF = tempDF.set_index(['public_client_id', 'Model']).pivot(columns='Modeling')['BaseProtBMI']
tempDF = tempDF.reset_index()
tempL = ['Female', 'Male', 'Both sex']
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Previous', y='Current',
               hue='Model', hue_order=tempL, palette='Set1', legend=False,
               col='Model', col_order=tempL, col_wrap=3,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, model in zip(p.axes.ravel(), tempL):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Model']==model]['Previous']
    tempS2 = tempDF.loc[tempDF['Model']==model]['Current']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
p.set_titles('{col_name}')
p.set_axis_labels('10 model predictions [kg/m'+r'$^2$'+']',
                  '1 model predictions [kg/m'+r'$^2$'+']')
plt.show()
print('')

print('Time-series biological BMI:')
#Plot difference b/w sex-combined and sex-stratified models for the time-series
tempDF = pd.concat([tsDF_F, tsDF_M, tsDF_B], axis=0)
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(tsDF_F), len(tsDF_M), len(tsDF_B)])
tempDF = tempDF.reset_index().set_index(['KeyIndex', 'Sex']).pivot(columns='Model')['ProtBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()

### 2-5. Baseline obesity classification

In [ ]:
#Sex-combined model
print('Sex-combined model')
tempDF = baseDF_B#Not copy just rename
for col_n in ['BaseBMI', 'BaseProtBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseProtBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempS = tempDF1['BaseProtBMI_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_B = pd.merge(tsDF_B, tempDF[['public_client_id', 'BaseBMI_class', 'BaseProtBMI_class']],
                  on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
tsDF_B = tsDF_B[['public_client_id', 'days_in_program', 'Season', 'log_ProtBMI', 'ProtBMI',
                 'log_BaseProtBMI', 'BaseProtBMI', 'BaseProtBMI_class',
                 'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_B)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-protBMI-BothSex.csv'
tsDF_B.to_csv(fileDir+fileName, index=True)

In [ ]:
#Sex-stratified model
print('Sex-stratified model')
tempDF = pd.concat([baseDF_F, baseDF_M], axis=0)
for col_n in ['BaseBMI', 'BaseProtBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseProtBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = pd.concat([tempDF1, tempDF2], axis=0)
col_n = 'BaseProtBMI'
tempL = []
for value in tempDF1[col_n].tolist():
    if np.isnan(value):
        tempL.append('NotCalculated')
    elif value < 18.5:
        tempL.append('Underweight')
    elif value < 25:
        tempL.append('Normal')
    elif value < 30:
        tempL.append('Overweight')
    elif value >= 30:
        tempL.append('Obese')
    else:#Just in case
        tempL.append('Error?')
tempDF1[col_n+'_class'] = tempL
tempS = tempDF1[col_n+'_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_FM = pd.concat([tsDF_F, tsDF_M], axis=0)
tsDF_FM = pd.merge(tsDF_FM, tempDF[['public_client_id', 'BaseBMI_class', 'BaseProtBMI_class']],
                   on='public_client_id', how='left')
tsDF_FM['KeyIndex'] = tsDF_FM['public_client_id'] + '_day' + tsDF_FM['days_in_program'].astype(str)
tsDF_FM = tsDF_FM.set_index('KeyIndex')
tsDF_FM = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'log_ProtBMI', 'ProtBMI',
                   'log_BaseProtBMI', 'BaseProtBMI', 'BaseProtBMI_class',
                   'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                   'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_FM)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-protBMI-FemaleMale.csv'
tsDF_FM.to_csv(fileDir+fileName, index=True)

## 3. Clinical labs

### 3-1. Prepare the baseline and sex-dependent DFs for LASSO

In [ ]:
#Import the cleaned time-series DF
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_time-series-chemDF-with-RF-imputation.tsv'
tsDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF)

In [ ]:
#Prepare the baseline DF for LASSO
tempDF = tsDF.sort_values(by=['public_client_id', 'days_in_program'], ascending=True)
baseDF = tempDF.drop_duplicates('public_client_id', keep='first')
display(baseDF)

In [ ]:
#Split DF for sex-dependent models
baseDF_F = baseDF.loc[baseDF['Sex']=='F']
baseDF_M = baseDF.loc[baseDF['Sex']=='M']
baseDF_B = baseDF#Not copy just rename
print('Baseline DF nrows (unique ID): (Female, Male, Both) = (',
      len(baseDF_F), ',', len(baseDF_M), ',', len(baseDF_B), ')')
tsDF_F = tsDF.loc[tsDF['Sex']=='F']
tsDF_M = tsDF.loc[tsDF['Sex']=='M']
tsDF_B = tsDF#Not copy just rename
print('Time-series DF nrows: (Female, Male, Both) = (',
      len(tsDF_F), ',', len(tsDF_M), ',', len(tsDF_B), ')')
print('Time-series DF unique ID: (Female, Male, Both) = (',
      len(tsDF_F['public_client_id'].unique()), ',',
      len(tsDF_M['public_client_id'].unique()), ',',
      len(tsDF_B['public_client_id'].unique()), ')')

### 3-2. Standarization with the baseline DF

In [ ]:
#Female cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_F.loc[:, ~baseDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_F.loc[:, ~tsDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Male cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_M.loc[:, ~baseDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_M.loc[:, ~tsDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_B.loc[:, ~baseDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_B.loc[:, ~tsDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 3-3. A cross-validated LASSO model trained with the baseline measurement

> For the baseline analyses, 10 LASSO models with 10 cross-validation were generated from 10 splitted DF.  
> ***–> Because it is difficult to manage the prediction difference b/w models and b/w participants, generate only 1 LASSO model with 10 CV.***  
> <– There are participants with only the baseline measurement, which would relieve the overfitting for the cohort in time-series analyses.  

In [ ]:
#Female model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_F.drop(columns=list(set(baseDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_F_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_F = model.score(baseDF_F_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_F_zx), index=tempDF.index, name='log_BaseChemBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseChemBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_F = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_F[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_F)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_F_zx), index=tsDF_F.index, name='log_ChemBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ChemBMI')
tempDF = tsDF_F[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_F = pd.merge(tempDF, baseDF_F, on='public_client_id', how='left')
tsDF_F['KeyIndex'] = tsDF_F['public_client_id'] + '_day' + tsDF_F['days_in_program'].astype(str)
tsDF_F = tsDF_F.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_F)

In [ ]:
#Male model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_M.drop(columns=list(set(baseDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_M_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_M = model.score(baseDF_M_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_M_zx), index=tempDF.index, name='log_BaseChemBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseChemBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_M = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_M[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_M)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_M_zx), index=tsDF_M.index, name='log_ChemBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ChemBMI')
tempDF = tsDF_M[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_M = pd.merge(tempDF, baseDF_M, on='public_client_id', how='left')
tsDF_M['KeyIndex'] = tsDF_M['public_client_id'] + '_day' + tsDF_M['days_in_program'].astype(str)
tsDF_M = tsDF_M.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_M)

In [ ]:
#Both sex model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_B.drop(columns=list(set(baseDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_B_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_B = model.score(baseDF_B_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_B_zx), index=tempDF.index, name='log_BaseChemBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseChemBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_B = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_B[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_B)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_B_zx), index=tsDF_B.index, name='log_ChemBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='ChemBMI')
tempDF = tsDF_B[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_B = pd.merge(tempDF, baseDF_B, on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_B)

### 3-4. Prediction accuracy of LASSO model

In [ ]:
#Plot R2 score
tempDF = pd.DataFrame([R2_F, R2_M, R2_B], index=['Female', 'Male', 'Both sex'], columns=['BaseChemBMI'])
tempDF.index.set_names('Model', inplace=True)
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF.reset_index(), y='Model', x='BaseChemBMI',
            hue='Model', palette='Set1', dodge=False, edgecolor='black')
sns.despine()
plt.xlabel('In-sample '+r'$R^2$'+' in 1 LASSO model')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

print('Female model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_F)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_F['log_BaseBMI'], baseDF_F['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_F['BaseBMI'], baseDF_F['BaseChemBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_F[['log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseChemBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseChemBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_F[['log_ChemBMI', 'ChemBMI', 'log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Male model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_M)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_M['log_BaseBMI'], baseDF_M['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_M['BaseBMI'], baseDF_M['BaseChemBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_M[['log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseChemBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseChemBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_M[['log_ChemBMI', 'ChemBMI', 'log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Both sex model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_B)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_B['log_BaseBMI'], baseDF_B['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_B['BaseBMI'], baseDF_B['BaseChemBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_B[['log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseChemBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseChemBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_B[['log_ChemBMI', 'ChemBMI', 'log_BaseChemBMI', 'BaseChemBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

In [ ]:
print('Baseline biological BMI:')
#Plot all models for the baseline
tempDF = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseChemBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Biological BMI [kg/m'+r'$^2$'+']')
plt.ylabel('Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
#Plot difference b/w sex-combined and sex-stratified models
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF = tempDF.set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseChemBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()
print('')

#Plot difference b/w previous 10 models and current 1 model
print('Comparison b/w the previous 10 models and the current 1 model for the baseline:')
##Prepare the previous predictions
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv'
tempDF3 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])
tempDF = tempDF.reset_index()
tempDF1 = tempDF[['public_client_id', 'BaseBMI', 'Model', 'BaseChemBMI']]
##Prepare the current predictions
tempDF2 = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF2['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF2 = tempDF2.reset_index()
tempDF2 = tempDF2[['public_client_id', 'BaseBMI', 'Model', 'BaseChemBMI']]
##Merge
tempDF = pd.concat([tempDF1, tempDF2], axis=0)
tempDF['Modeling'] = np.repeat(['Previous', 'Current'], [len(tempDF1), len(tempDF2)])
##Plot1
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='BaseChemBMI', y='BaseBMI', hue='Modeling', palette='tab10',
               col='Model', col_wrap=3, scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
#p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax in p.axes.ravel():
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
p.set_titles('{col_name}')
p.set_axis_labels('Biological BMI [kg/m'+r'$^2$'+']', 'Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
##Plot2
tempDF = tempDF.set_index(['public_client_id', 'Model']).pivot(columns='Modeling')['BaseChemBMI']
tempDF = tempDF.reset_index()
tempL = ['Female', 'Male', 'Both sex']
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Previous', y='Current',
               hue='Model', hue_order=tempL, palette='Set1', legend=False,
               col='Model', col_order=tempL, col_wrap=3,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, model in zip(p.axes.ravel(), tempL):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Model']==model]['Previous']
    tempS2 = tempDF.loc[tempDF['Model']==model]['Current']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
p.set_titles('{col_name}')
p.set_axis_labels('10 model predictions [kg/m'+r'$^2$'+']',
                  '1 model predictions [kg/m'+r'$^2$'+']')
plt.show()
print('')

print('Time-series biological BMI:')
#Plot difference b/w sex-combined and sex-stratified models for the time-series
tempDF = pd.concat([tsDF_F, tsDF_M, tsDF_B], axis=0)
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(tsDF_F), len(tsDF_M), len(tsDF_B)])
tempDF = tempDF.reset_index().set_index(['KeyIndex', 'Sex']).pivot(columns='Model')['ChemBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()

### 3-5. Baseline obesity classification

In [ ]:
#Sex-combined model
print('Sex-combined model')
tempDF = baseDF_B#Not copy just rename
for col_n in ['BaseBMI', 'BaseChemBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseChemBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempS = tempDF1['BaseChemBMI_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_B = pd.merge(tsDF_B, tempDF[['public_client_id', 'BaseBMI_class', 'BaseChemBMI_class']],
                  on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
tsDF_B = tsDF_B[['public_client_id', 'days_in_program', 'Season', 'log_ChemBMI', 'ChemBMI',
                 'log_BaseChemBMI', 'BaseChemBMI', 'BaseChemBMI_class',
                 'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_B)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-chemBMI-BothSex.csv'
tsDF_B.to_csv(fileDir+fileName, index=True)

In [ ]:
#Sex-stratified model
print('Sex-stratified model')
tempDF = pd.concat([baseDF_F, baseDF_M], axis=0)
for col_n in ['BaseBMI', 'BaseChemBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseChemBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = pd.concat([tempDF1, tempDF2], axis=0)
col_n = 'BaseChemBMI'
tempL = []
for value in tempDF1[col_n].tolist():
    if np.isnan(value):
        tempL.append('NotCalculated')
    elif value < 18.5:
        tempL.append('Underweight')
    elif value < 25:
        tempL.append('Normal')
    elif value < 30:
        tempL.append('Overweight')
    elif value >= 30:
        tempL.append('Obese')
    else:#Just in case
        tempL.append('Error?')
tempDF1[col_n+'_class'] = tempL
tempS = tempDF1[col_n+'_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_FM = pd.concat([tsDF_F, tsDF_M], axis=0)
tsDF_FM = pd.merge(tsDF_FM, tempDF[['public_client_id', 'BaseBMI_class', 'BaseChemBMI_class']],
                   on='public_client_id', how='left')
tsDF_FM['KeyIndex'] = tsDF_FM['public_client_id'] + '_day' + tsDF_FM['days_in_program'].astype(str)
tsDF_FM = tsDF_FM.set_index('KeyIndex')
tsDF_FM = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'log_ChemBMI', 'ChemBMI',
                   'log_BaseChemBMI', 'BaseChemBMI', 'BaseChemBMI_class',
                   'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                   'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_FM)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-chemBMI-FemaleMale.csv'
tsDF_FM.to_csv(fileDir+fileName, index=True)

## 4. Combined omics

### 4-1. Prepare the baseline and sex-dependent DFs for LASSO

In [ ]:
#Import the cleaned time-series DF
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_time-series-combiDF-with-RF-imputation.tsv'
tsDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF)

In [ ]:
#Prepare the baseline DF for LASSO
tempDF = tsDF.sort_values(by=['public_client_id', 'days_in_program'], ascending=True)
baseDF = tempDF.drop_duplicates('public_client_id', keep='first')
display(baseDF)

In [ ]:
#Split DF for sex-dependent models
baseDF_F = baseDF.loc[baseDF['Sex']=='F']
baseDF_M = baseDF.loc[baseDF['Sex']=='M']
baseDF_B = baseDF#Not copy just rename
print('Baseline DF nrows (unique ID): (Female, Male, Both) = (',
      len(baseDF_F), ',', len(baseDF_M), ',', len(baseDF_B), ')')
tsDF_F = tsDF.loc[tsDF['Sex']=='F']
tsDF_M = tsDF.loc[tsDF['Sex']=='M']
tsDF_B = tsDF#Not copy just rename
print('Time-series DF nrows: (Female, Male, Both) = (',
      len(tsDF_F), ',', len(tsDF_M), ',', len(tsDF_B), ')')
print('Time-series DF unique ID: (Female, Male, Both) = (',
      len(tsDF_F['public_client_id'].unique()), ',',
      len(tsDF_M['public_client_id'].unique()), ',',
      len(tsDF_B['public_client_id'].unique()), ')')

### 4-2. Standarization with the baseline DF

In [ ]:
#Female cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_F.loc[:, ~baseDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_F.loc[:, ~tsDF_F.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_F_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_F_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Male cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_M.loc[:, ~baseDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_M.loc[:, ~tsDF_M.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_M_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_M_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

In [ ]:
#Both sex cohort
tempL = ['public_client_id', 'days_in_program', 'Season',
         'log_BaseBMI', 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

#Baseline DF
print('Baseline DF:')
tempDF = baseDF_B.loc[:, ~baseDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Compute the mean and std for Z-score transformation based on the baseline distribution
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
scaler.fit(tempDF)
##Z-score transformation of the baseline DF with the baseline distribution
tempA = scaler.transform(tempDF)
baseDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = baseDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()
print('')

#Time-series DF
print('Time-series DF:')
tempDF = tsDF_B.loc[:, ~tsDF_B.columns.isin(tempL)]
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()
##Z-score transformation of the time-series DF with the baseline distribution
tempA = scaler.transform(tempDF)#Already fitted with the baseline DF
tsDF_B_zx = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)
##Confirmation
tempDF = tsDF_B_zx#Not copy just rename
display(tempDF.describe())
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 4-3. A cross-validated LASSO model trained with the baseline measurement

> For the baseline analyses, 10 LASSO models with 10 cross-validation were generated from 10 splitted DF.  
> ***–> Because it is difficult to manage the prediction difference b/w models and b/w participants, generate only 1 LASSO model with 10 CV.***  
> <– There are participants with only the baseline measurement, which would relieve the overfitting for the cohort in time-series analyses.  

In [ ]:
#Female model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_F.drop(columns=list(set(baseDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_F_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_F = model.score(baseDF_F_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_F_zx), index=tempDF.index, name='log_BaseCombiBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseCombiBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_F = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_F[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_F)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_F_zx), index=tsDF_F.index, name='log_CombiBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='CombiBMI')
tempDF = tsDF_F[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_F = pd.merge(tempDF, baseDF_F, on='public_client_id', how='left')
tsDF_F['KeyIndex'] = tsDF_F['public_client_id'] + '_day' + tsDF_F['days_in_program'].astype(str)
tsDF_F = tsDF_F.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_F)

In [ ]:
#Male model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_M.drop(columns=list(set(baseDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_M_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_M = model.score(baseDF_M_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_M_zx), index=tempDF.index, name='log_BaseCombiBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseCombiBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_M = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_M[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_M)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_M_zx), index=tsDF_M.index, name='log_CombiBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='CombiBMI')
tempDF = tsDF_M[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_M = pd.merge(tempDF, baseDF_M, on='public_client_id', how='left')
tsDF_M['KeyIndex'] = tsDF_M['public_client_id'] + '_day' + tsDF_M['days_in_program'].astype(str)
tsDF_M = tsDF_M.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_M)

In [ ]:
#Both sex model

#Prepare the dependent variable and the model parameters
tempDF = baseDF_B.drop(columns=list(set(baseDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True, normalize=False, precompute='auto', cv=10)

#Train a cross-validated LASSO model with the baseline DF
t_start = time.time()
model.fit(baseDF_B_zx, tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time for a 10-fold CV LASSO model:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Evaluation as reference
R2_B = model.score(baseDF_B_zx, tempDF)

#Prediction with the trained model for the baseline DF
tempS1 = pd.Series(model.predict(baseDF_B_zx), index=tempDF.index, name='log_BaseCombiBMI')

#Convert to original scale and clean the result of baseline DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='BaseCombiBMI')
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']
baseDF_B = pd.concat([tempS1, tempS2, tempDF,
                      baseDF_B[['public_client_id',
                                'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]],
                     axis=1, ignore_index=False)
print('Baseline DF:')
display(baseDF_B)

#Prediction with the trained model for the time-series DF
tempS1 = pd.Series(model.predict(tsDF_B_zx), index=tsDF_B.index, name='log_CombiBMI')

#Convert to original scale and clean the result of time-series DF
tempS2 = pd.Series(np.e**tempS1, index=tempS1.index, name='CombiBMI')
tempDF = tsDF_B[['public_client_id', 'days_in_program', 'Season']]
tempDF = pd.concat([tempDF, tempS1, tempS2], axis=1, ignore_index=False)
tsDF_B = pd.merge(tempDF, baseDF_B, on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
print('Time-series DF:')
display(tsDF_B)

### 4-4. Prediction accuracy of LASSO model

In [ ]:
#Plot R2 score
tempDF = pd.DataFrame([R2_F, R2_M, R2_B], index=['Female', 'Male', 'Both sex'], columns=['BaseCombiBMI'])
tempDF.index.set_names('Model', inplace=True)
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF.reset_index(), y='Model', x='BaseCombiBMI',
            hue='Model', palette='Set1', dodge=False, edgecolor='black')
sns.despine()
plt.xlabel('In-sample '+r'$R^2$'+' in 1 LASSO model')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

print('Female model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_F)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_F['log_BaseBMI'], baseDF_F['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_F['BaseBMI'], baseDF_F['BaseCombiBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_F[['log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_F[['log_CombiBMI', 'CombiBMI', 'log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Male model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_M)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_M['log_BaseBMI'], baseDF_M['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_M['BaseBMI'], baseDF_M['BaseCombiBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_M[['log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_M[['log_CombiBMI', 'CombiBMI', 'log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

print('Both sex model')
print('• In-sample R2 Score in the LASSO model with the baseline measurement:', R2_B)
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(baseDF_B['log_BaseBMI'], baseDF_B['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(baseDF_B['BaseBMI'], baseDF_B['BaseCombiBMI']))#output: Pearson's r, p-value
print('• Summary of the baseline DF:')
display(baseDF_B[['log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('• Cf. Previous 10 LASSO models for baseline analyses')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv'
tempDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI (log-scale):',
      stats.pearsonr(tempDF['log_BaseBMI'], tempDF['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Pearson\'s r b/w observed vs. LASSO-predicted BMI:',
      stats.pearsonr(tempDF['BaseBMI'], tempDF['BaseCombiBMI']))#output: Pearson's r, p-value
print('  - Summary of the baseline DF:')
display(tempDF.describe())
print('• Cf. Summary of the time-series DF:')
display(tsDF_B[['log_CombiBMI', 'CombiBMI', 'log_BaseCombiBMI', 'BaseCombiBMI', 'log_BaseBMI', 'BaseBMI']].describe())
print('')

In [ ]:
print('Baseline biological BMI:')
#Plot all models for the baseline
tempDF = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseCombiBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Biological BMI [kg/m'+r'$^2$'+']')
plt.ylabel('Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
#Plot difference b/w sex-combined and sex-stratified models
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF = tempDF.set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseCombiBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()
print('')

#Plot difference b/w previous 10 models and current 1 model
print('Comparison b/w the previous 10 models and the current 1 model for the baseline:')
##Prepare the previous predictions
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv'
tempDF3 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF = pd.concat([tempDF1, tempDF2, tempDF3], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(tempDF1), len(tempDF2), len(tempDF3)])
tempDF = tempDF.reset_index()
tempDF1 = tempDF[['public_client_id', 'BaseBMI', 'Model', 'BaseCombiBMI']]
##Prepare the current predictions
tempDF2 = pd.concat([baseDF_F, baseDF_M, baseDF_B], axis=0)
tempDF2['Model'] = np.repeat(['Female', 'Male', 'Both sex'], [len(baseDF_F), len(baseDF_M), len(baseDF_B)])
tempDF2 = tempDF2.reset_index()
tempDF2 = tempDF2[['public_client_id', 'BaseBMI', 'Model', 'BaseCombiBMI']]
##Merge
tempDF = pd.concat([tempDF1, tempDF2], axis=0)
tempDF['Modeling'] = np.repeat(['Previous', 'Current'], [len(tempDF1), len(tempDF2)])
##Plot1
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='BaseCombiBMI', y='BaseBMI', hue='Modeling', palette='tab10',
               col='Model', col_wrap=3, scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
#p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax in p.axes.ravel():
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
p.set_titles('{col_name}')
p.set_axis_labels('Biological BMI [kg/m'+r'$^2$'+']', 'Measured BMI [kg/m'+r'$^2$'+']')
plt.show()
##Plot2
tempDF = tempDF.set_index(['public_client_id', 'Model']).pivot(columns='Modeling')['BaseCombiBMI']
tempDF = tempDF.reset_index()
tempL = ['Female', 'Male', 'Both sex']
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Previous', y='Current',
               hue='Model', hue_order=tempL, palette='Set1', legend=False,
               col='Model', col_order=tempL, col_wrap=3,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, model in zip(p.axes.ravel(), tempL):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Model']==model]['Previous']
    tempS2 = tempDF.loc[tempDF['Model']==model]['Current']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
p.set_titles('{col_name}')
p.set_axis_labels('10 model predictions [kg/m'+r'$^2$'+']',
                  '1 model predictions [kg/m'+r'$^2$'+']')
plt.show()
print('')

print('Time-series biological BMI:')
#Plot difference b/w sex-combined and sex-stratified models for the time-series
tempDF = pd.concat([tsDF_F, tsDF_M, tsDF_B], axis=0)
tempDF['Model'] = np.repeat(['Sex-stratified', 'Sex-stratified', 'Sex-combined'],
                            [len(tsDF_F), len(tsDF_M), len(tsDF_B)])
tempDF = tempDF.reset_index().set_index(['KeyIndex', 'Sex']).pivot(columns='Model')['CombiBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='notebook')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='medium')
    #Change facet label
    ax.set_title(tempD[sex])
p.set_axis_labels('Sex-stratified model [kg/m'+r'$^2$'+']',
                  'Sex-combined model [kg/m'+r'$^2$'+']')
plt.show()

### 4-5. Baseline obesity classification

In [ ]:
#Sex-combined model
print('Sex-combined model')
tempDF = baseDF_B#Not copy just rename
for col_n in ['BaseBMI', 'BaseCombiBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseCombiBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempS = tempDF1['BaseCombiBMI_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_B = pd.merge(tsDF_B, tempDF[['public_client_id', 'BaseBMI_class', 'BaseCombiBMI_class']],
                  on='public_client_id', how='left')
tsDF_B['KeyIndex'] = tsDF_B['public_client_id'] + '_day' + tsDF_B['days_in_program'].astype(str)
tsDF_B = tsDF_B.set_index('KeyIndex')
tsDF_B = tsDF_B[['public_client_id', 'days_in_program', 'Season', 'log_CombiBMI', 'CombiBMI',
                 'log_BaseCombiBMI', 'BaseCombiBMI', 'BaseCombiBMI_class',
                 'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                 'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_B)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-combiBMI-BothSex.csv'
tsDF_B.to_csv(fileDir+fileName, index=True)

In [ ]:
#Sex-stratified model
print('Sex-stratified model')
tempDF = pd.concat([baseDF_F, baseDF_M], axis=0)
for col_n in ['BaseBMI', 'BaseCombiBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')
#Check the previous classification
print('Previous 10 model-predicted BaseCombiBMI_class:')
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv'
tempDF1 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv'
tempDF2 = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = pd.concat([tempDF1, tempDF2], axis=0)
col_n = 'BaseCombiBMI'
tempL = []
for value in tempDF1[col_n].tolist():
    if np.isnan(value):
        tempL.append('NotCalculated')
    elif value < 18.5:
        tempL.append('Underweight')
    elif value < 25:
        tempL.append('Normal')
    elif value < 30:
        tempL.append('Overweight')
    elif value >= 30:
        tempL.append('Obese')
    else:#Just in case
        tempL.append('Error?')
tempDF1[col_n+'_class'] = tempL
tempS = tempDF1[col_n+'_class'].value_counts()
tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF1)*100})
display(tempDF1)
print('')

#Clean and save the time-series DF
tsDF_FM = pd.concat([tsDF_F, tsDF_M], axis=0)
tsDF_FM = pd.merge(tsDF_FM, tempDF[['public_client_id', 'BaseBMI_class', 'BaseCombiBMI_class']],
                   on='public_client_id', how='left')
tsDF_FM['KeyIndex'] = tsDF_FM['public_client_id'] + '_day' + tsDF_FM['days_in_program'].astype(str)
tsDF_FM = tsDF_FM.set_index('KeyIndex')
tsDF_FM = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'log_CombiBMI', 'CombiBMI',
                   'log_BaseCombiBMI', 'BaseCombiBMI', 'BaseCombiBMI_class',
                   'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
                   'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF_FM)
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-combiBMI-FemaleMale.csv'
tsDF_FM.to_csv(fileDir+fileName, index=True)